Import potrzebnych bibliotek

In [147]:
import pandas as pd
import numpy as np
import time

Wczytanie danych

In [148]:
df = pd.read_csv("auction_results_color_svd.csv")

df.head()

,ARTIST,TECHNIQUE,SIGNATURE,CONDITION,TOTAL DIMENSIONS,YEAR,Colorfulness Score,SVD Entropy,PRICE
0,218,-1.295300,1,2,-0.157723,-1.039766,51.632554,5.453204,150
1,101,-0.122087,2,2,-0.442668,-0.580107,161.631656,6.154763,270
2,274,-0.122087,2,2,0.263423,-0.626073,117.464780,6.908661,360
3,354,3.397553,0,2,-0.827075,-0.488176,164.609302,6.986244,343
4,354,-0.122087,2,2,-0.145178,0.431142,91.023011,5.859255,150


In [149]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22110 entries, 0 to 22109
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ARTIST              22110 non-null  int64  
 1   TECHNIQUE           22110 non-null  float64
 2   SIGNATURE           22110 non-null  int64  
 3   CONDITION           22110 non-null  int64  
 4   TOTAL DIMENSIONS    22110 non-null  float64
 5   YEAR                22110 non-null  float64
 6   Colorfulness Score  22110 non-null  float64
 7   SVD Entropy         22110 non-null  float64
 8   PRICE               22110 non-null  int64  
dtypes: float64(5), int64(4)
memory usage: 1.5 MB


Dopasowanie typów zmiennych

In [150]:
zmienne_kategoryczne =  ['ARTIST', 'TECHNIQUE', 'SIGNATURE', 'CONDITION']

bloki_kategoryczne = []
for zmienna in zmienne_kategoryczne:
    kody = df[zmienna].astype('category').cat.codes.to_numpy(dtype=np.int32)
    liczba_klas = np.max(kody) + 1
    one_hot = np.eye(liczba_klas)[kody]
    bloki_kategoryczne.append(one_hot)


X_kat_cale = np.hstack(bloki_kategoryczne)


Dane treningowe i testowe

In [151]:
np.random.seed(42)
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

indeksy_train = df_train.index
indeksy_test = df_test.index

X_kat_train = X_kat_cale[indeksy_train]
X_kat_test = X_kat_cale[indeksy_test]

Normalizacja zmiennych

In [152]:
zmienne_liczbowe = ["TOTAL DIMENSIONS", "YEAR", "Colorfulness Score", "SVD Entropy"]
srednia = df_train[zmienne_liczbowe].mean()
odchylenie = df_train[zmienne_liczbowe].std()

df_train[zmienne_liczbowe] = (df_train[zmienne_liczbowe] - srednia) / odchylenie
df_test[zmienne_liczbowe] = (df_test[zmienne_liczbowe] - srednia) / odchylenie

Przejście na tablice NumPy

In [153]:
X_num_train = df_train[zmienne_liczbowe].to_numpy(dtype=np.float32)
X_num_test = df_test[zmienne_liczbowe].to_numpy(dtype=np.float32)

X_train = np.hstack([X_kat_train, X_num_train], dtype=np.float32)
X_test = np.hstack([X_kat_test, X_num_test], dtype=np.float32)


In [154]:
# --- PRZEŁĄCZNIK METODY ---
# Ustaw na False, by użyć starej metody (Z-score). 
# Ustaw na True, by użyć nowej metody (Transformacja logarytmiczna).
UZYJ_LOGARYTMU = True
# --------------------------

if UZYJ_LOGARYTMU:
    print("Używamy transformacji logarytmicznej (zapobiega ujemnym cenom).")
    # Zamieniamy ceny na ich logarytmy
    y_train = np.log(df_train['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)
    y_test = np.log(df_test['PRICE']).to_numpy(dtype=np.float32).reshape(-1, 1)
    
    # Funkcja odwracająca to po prostu funkcja eksponencjalna (odwrotność logarytmu)
    def prawdziwa_cena(wyliczona_wartosc):
        return np.exp(wyliczona_wartosc)
        
else:
    print("Używamy normalizacji Z-score (może generować ujemne ceny).")
    # Stara metoda
    srednia_cena = df_train['PRICE'].mean()
    odchylenie_cena = df_train['PRICE'].std()
    
    y_train = ((df_train['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
    y_test = ((df_test['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
    
    # Stara funkcja odwracająca
    def prawdziwa_cena(znormalizowana_cena):
        return znormalizowana_cena * odchylenie_cena + srednia_cena

Używamy transformacji logarytmicznej (zapobiega ujemnym cenom).


Funkcja do obliczania prawdziwych cen

Tworzenie Dense Layer

In [155]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))
    
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.dot(inputs, self.weights) + self.biases
        
    def backward(self, dvalues):
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)

Funkcje aktywacji

In [156]:
class Activation_ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

class Activation_Linear:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = inputs
    
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()

Funkcja straty

In [157]:
class Loss:
    def calculate(self, output, y):
        sample_losses = self.forward(output, y)
        data_loss = np.mean(sample_losses)
        return data_loss
    
class Loss_MSE(Loss):
    def forward(self, y_pred, y_true):
        sample_losses = np.mean((y_pred - y_true) ** 2, axis=-1)
        return sample_losses
    
    def backward(self, dvalues, y_true):
        liczba_probek = len(dvalues)
        self.dinputs = -2 * (y_true - dvalues) / liczba_probek



Optymalizator SGD

In [158]:
class Optimizer_SGD:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update_params(self, layer):
        layer.weights -= self.learning_rate * layer.dweights
        layer.biases -= self.learning_rate * layer.dbiases

Inicjalizacja sieci i optymalizatora

In [159]:
# liczba_cech = X_train.shape[1]

# dense1 = Layer_Dense(liczba_cech, 64)
# activation1 = Activation_ReLU()

# dense2 = Layer_Dense(64, 32)
# activation2 = Activation_ReLU()

# dense3 = Layer_Dense(32, 1)
# activation3 = Activation_Linear()

# loss_function = Loss_MSE()


# optimizer = Optimizer_SGD(learning_rate=0.1)

Badanie liczby neuronów

In [160]:
# # --- USTAWIENIA BADANIA ---
# # 4 różne warianty liczby neuronów (warstwa1, warstwa2)
# architektury_do_testu = [
#     (16, 8),
#     (32, 16),
#     (64, 32),
#     (128, 64),
#     (256, 128)
# ]
# powtorzenia = 3       # Ile razy powtarzamy uczenie dla każdego wariantu
# epoki = 100           # Dla oszczędności czasu na testach, możesz dać np. 100 lub 150
# batch_size = 256
# learning_rate = 0.1  # Stały LR dla tego testu

# wyniki_raport = []
# liczba_cech = X_train.shape[1]

# print("="*60)
# print("ROZPOCZYNAMY BADANIE: Rozmiar warstw ukrytych")
# print("="*60)

# for arch in architektury_do_testu:
#     n1, n2 = arch
#     print(f"\n---> Testuję architekturę: [{n1}, {n2}] Pętla powtórzeń: ", end="")
    
#     mae_train_historia = []
#     mae_test_historia = []
    
#     start_time = time.time()
    
#     for powtorzenie in range(powtorzenia):
#         print(f"{powtorzenie+1}...", end=" ")
        
#         # 1. ZAWSZE INICJUJEMY SIEĆ OD NOWA W ŚRODKU PĘTLI! (bardzo ważne)
#         dense1 = Layer_Dense(liczba_cech, n1)
#         activation1 = Activation_ReLU()
        
#         dense2 = Layer_Dense(n1, n2)
#         activation2 = Activation_ReLU()
        
#         dense3 = Layer_Dense(n2, 1)
#         activation3 = Activation_Linear()
        
#         loss_function = Loss_MSE()
#         optimizer = Optimizer_SGD(learning_rate=learning_rate)
        
#         # 2. TRENING (Szybka pętla batchowa)
#         for epoch in range(epoki):
#             for start_idx in range(0, len(X_train), batch_size):
#                 end_idx = start_idx + batch_size
#                 X_batch = X_train[start_idx:end_idx]
#                 y_batch = y_train[start_idx:end_idx]
                
#                 # Forward
#                 dense1.forward(X_batch)
#                 activation1.forward(dense1.output)
#                 dense2.forward(activation1.output)
#                 activation2.forward(dense2.output)
#                 dense3.forward(activation2.output)
#                 activation3.forward(dense3.output)
                
#                 # Błąd i Backward
#                 loss_function.backward(activation3.output, y_batch)
#                 activation3.backward(loss_function.dinputs)
#                 dense3.backward(activation3.dinputs)
#                 activation2.backward(dense3.dinputs)
#                 dense2.backward(activation2.dinputs)
#                 activation1.backward(dense2.dinputs)
#                 dense1.backward(activation1.dinputs)
                
#                 # Aktualizacja
#                 optimizer.update_params(dense1)
#                 optimizer.update_params(dense2)
#                 optimizer.update_params(dense3)

#         # 3. EWALUACJA NA ZBIORZE TRENINGOWYM
#         dense1.forward(X_train)
#         activation1.forward(dense1.output)
#         dense2.forward(activation1.output)
#         activation2.forward(dense2.output)
#         dense3.forward(activation2.output)
#         activation3.forward(dense3.output)
        
#         wymyslone_train_dolary = prawdziwa_cena(activation3.output)
#         prawdziwe_train_dolary = prawdziwa_cena(y_train)
#         mae_train = np.mean(np.abs(prawdziwe_train_dolary - wymyslone_train_dolary))
#         mae_train_historia.append(mae_train)

#         # 4. EWALUACJA NA ZBIORZE TESTOWYM
#         dense1.forward(X_test)
#         activation1.forward(dense1.output)
#         dense2.forward(activation1.output)
#         activation2.forward(dense2.output)
#         dense3.forward(activation2.output)
#         activation3.forward(dense3.output)
        
#         wymyslone_test_dolary = prawdziwa_cena(activation3.output)
#         prawdziwe_test_dolary = prawdziwa_cena(y_test)
#         mae_test = np.mean(np.abs(prawdziwe_test_dolary - wymyslone_test_dolary))
#         mae_test_historia.append(mae_test)
        
#     czas_trwania = time.time() - start_time
#     print(f"Zakończono w {czas_trwania:.1f} s.")

#     # 5. AGREGACJA WYNIKÓW DLA DANEJ ARCHITEKTURY
#     wyniki_raport.append({
#         "Architektura": f"[{n1}, {n2}]",
#         "Śr. MAE (Train) [$]": round(np.mean(mae_train_historia), 2),
#         "Śr. MAE (Test) [$]": round(np.mean(mae_test_historia), 2),
#         "Najlepsze MAE (Test) [$]": round(np.min(mae_test_historia), 2)
#     })

# # --- WYSWIETLANIE TABELKI ---
# print("\n" + "="*60)
# print("PODSUMOWANIE WYNIKÓW: WPŁYW ARCHITEKTURY")
# print("="*60)
# df_raport = pd.DataFrame(wyniki_raport)
# print(df_raport.to_string(index=False))

Badanie Learning rate

In [161]:

# --- USTAWIENIA BADANIA: KROK 2 (Learning Rate) ---
# Zamrażamy najlepszą architekturę z poprzedniego kroku:
n1, n2 = 128, 64

# 4 warianty współczynnika uczenia do przetestowania:
learning_rates_do_testu = [0.1, 0.05, 0.01, 0.001]

powtorzenia = 3       
epoki = 100           
batch_size = 256      

wyniki_raport_lr = []
liczba_cech = X_train.shape[1]

print("="*60)
print(f"ROZPOCZYNAMY BADANIE: Współczynnik uczenia (Sieć: [{n1}, {n2}])")
print("="*60)

for lr in learning_rates_do_testu:
    print(f"\n---> Testuję Learning Rate: {lr} Pętla powtórzeń: ", end="")
    
    mae_train_historia = []
    mae_test_historia = []
    
    start_time = time.time()
    
    for powtorzenie in range(powtorzenia):
        print(f"{powtorzenie+1}...", end=" ")
        
        # 1. ZAWSZE INICJUJEMY SIEĆ OD NOWA
        dense1 = Layer_Dense(liczba_cech, n1)
        activation1 = Activation_ReLU()
        
        dense2 = Layer_Dense(n1, n2)
        activation2 = Activation_ReLU()
        
        dense3 = Layer_Dense(n2, 1)
        activation3 = Activation_Linear()
        
        loss_function = Loss_MSE()
        # TUTAJ PODMIENIAMY LEARNING RATE:
        optimizer = Optimizer_SGD(learning_rate=lr) 
        
        # 2. TRENING 
        for epoch in range(epoki):
            for start_idx in range(0, len(X_train), batch_size):
                end_idx = start_idx + batch_size
                X_batch = X_train[start_idx:end_idx]
                y_batch = y_train[start_idx:end_idx]
                
                # Forward
                dense1.forward(X_batch)
                activation1.forward(dense1.output)
                dense2.forward(activation1.output)
                activation2.forward(dense2.output)
                dense3.forward(activation2.output)
                activation3.forward(dense3.output)
                
                # Błąd i Backward
                loss_function.backward(activation3.output, y_batch)
                activation3.backward(loss_function.dinputs)
                dense3.backward(activation3.dinputs)
                activation2.backward(dense3.dinputs)
                dense2.backward(activation2.dinputs)
                activation1.backward(dense2.dinputs)
                dense1.backward(activation1.dinputs)
                
                # Aktualizacja
                optimizer.update_params(dense1)
                optimizer.update_params(dense2)
                optimizer.update_params(dense3)

        # 3. EWALUACJA NA ZBIORZE TRENINGOWYM
        dense1.forward(X_train)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_train_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_train_dolary = prawdziwa_cena(y_train)
        mae_train = np.mean(np.abs(prawdziwe_train_dolary - wymyslone_train_dolary))
        mae_train_historia.append(mae_train)

        # 4. EWALUACJA NA ZBIORZE TESTOWYM
        dense1.forward(X_test)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        activation2.forward(dense2.output)
        dense3.forward(activation2.output)
        activation3.forward(dense3.output)
        
        wymyslone_test_dolary = prawdziwa_cena(activation3.output)
        prawdziwe_test_dolary = prawdziwa_cena(y_test)
        mae_test = np.mean(np.abs(prawdziwe_test_dolary - wymyslone_test_dolary))
        mae_test_historia.append(mae_test)
        
    czas_trwania = time.time() - start_time
    print(f"Zakończono w {czas_trwania:.1f} s.")

    # 5. AGREGACJA WYNIKÓW
    wyniki_raport_lr.append({
        "Learning Rate": lr,
        "Śr. MAE (Train) [$]": round(np.mean(mae_train_historia), 2),
        "Śr. MAE (Test) [$]": round(np.mean(mae_test_historia), 2),
        "Najlepsze MAE (Test) [$]": round(np.min(mae_test_historia), 2)
    })

# --- WYSWIETLANIE TABELKI ---
print("\n" + "="*60)
print("PODSUMOWANIE WYNIKÓW: WPŁYW WSPÓŁCZYNNIKA UCZENIA (LR)")
print("="*60)
df_raport_lr = pd.DataFrame(wyniki_raport_lr)
print(df_raport_lr.to_string(index=False))

ROZPOCZYNAMY BADANIE: Współczynnik uczenia (Sieć: [128, 64])

---> Testuję Learning Rate: 0.1 Pętla powtórzeń: 1... 2... 3... Zakończono w 23.1 s.

---> Testuję Learning Rate: 0.05 Pętla powtórzeń: 1... 2... 3... Zakończono w 22.9 s.

---> Testuję Learning Rate: 0.01 Pętla powtórzeń: 1... 2... 3... Zakończono w 22.9 s.

---> Testuję Learning Rate: 0.001 Pętla powtórzeń: 1... 2... 3... Zakończono w 22.6 s.

PODSUMOWANIE WYNIKÓW: WPŁYW WSPÓŁCZYNNIKA UCZENIA (LR)
 Learning Rate  Śr. MAE (Train) [$]  Śr. MAE (Test) [$]  Najlepsze MAE (Test) [$]
         0.100               106.10              113.04                    107.31
         0.050               105.65              108.82                    107.70
         0.010               109.67              112.43                    111.80
         0.001               158.84              158.71                    157.82


Pętla ucząca

In [162]:
# epoki = 100           
# batch_size = 256      

# print("Rozpoczynamy szybki trening na paczkach!\n" + "-"*30)

# for epoch in range(epoki + 1):
    
#     loss_epoki = 0
#     ilosc_paczek = 0
    
    
#     for start_idx in range(0, len(X_train), batch_size):
        
        
#         end_idx = start_idx + batch_size
#         X_batch = X_train[start_idx:end_idx]
#         y_batch = y_train[start_idx:end_idx]
        
        
#         dense1.forward(X_batch)
#         activation1.forward(dense1.output)
        
#         dense2.forward(activation1.output)
#         activation2.forward(dense2.output)
        
#         dense3.forward(activation2.output)
#         activation3.forward(dense3.output)

        
#         loss = loss_function.calculate(activation3.output, y_batch)
#         loss_epoki += loss
#         ilosc_paczek += 1

        
#         loss_function.backward(activation3.output, y_batch)
        
#         activation3.backward(loss_function.dinputs)
#         dense3.backward(activation3.dinputs)
        
#         activation2.backward(dense3.dinputs)
#         dense2.backward(activation2.dinputs)
        
#         activation1.backward(dense2.dinputs)
#         dense1.backward(activation1.dinputs)

        
#         optimizer.update_params(dense1)
#         optimizer.update_params(dense2)
#         optimizer.update_params(dense3)

    
#     sredni_blad_mse = loss_epoki / ilosc_paczek
    
#     if epoch % 10 == 0:
#         print(f"Epoka: {epoch:3} | Średni błąd (MSE): {sredni_blad_mse:.5f}")

# print("-" * 30 + "\nTrening zakończony!")

In [163]:
# print(f"Średni błąd MSE na ostatniej epoce: {sredni_blad_mse:.5f}")
# print(prawdziwa_cena(sredni_blad_mse))

In [164]:
# print("=== TESTOWANIE SIECI NA NIEZNANYCH DANYCH (X_test) ===\n")

# dense1.forward(X_test)
# activation1.forward(dense1.output)

# dense2.forward(activation1.output)
# activation2.forward(dense2.output)

# dense3.forward(activation2.output)
# activation3.forward(dense3.output)

# test_loss = loss_function.calculate(activation3.output, y_test)

# print(f"Błąd (MSE) na zbiorze TRENINGOWYM wynosił: {sredni_blad_mse:.5f}")
# print(f"Błąd (MSE) na zbiorze TESTOWYM wynosi:       {test_loss:.5f}")
# print("-" * 60)


# print("5 przykładowych wycen dla zupełnie nowych obrazów:\n")

# for i in range(5):
    
#     wymyslona_znormalizowana = activation3.output[i].item()
#     prawdziwa_znormalizowana = y_test[i].item()
    
#     wymyslona_prawdziwa = prawdziwa_cena(wymyslona_znormalizowana)
#     prawdziwa_w_dolarach = prawdziwa_cena(prawdziwa_znormalizowana)
    
#     pomylka = abs(prawdziwa_w_dolarach - wymyslona_prawdziwa)
    
#     print(f"Obraz {i+1} | Wyliczona: {wymyslona_prawdziwa:12.2f} | Prawdziwa: {prawdziwa_w_dolarach:12.2f} | Pomyłka: {pomylka:10.2f}")

In [165]:
# wymyslone_ceny_dolary = prawdziwa_cena(activation3.output)
# prawdziwe_ceny_dolary = prawdziwa_cena(y_test)

# mae_w_dolarach = np.mean(np.abs(prawdziwe_ceny_dolary - wymyslone_ceny_dolary))

# print(f"\nOSTATECZNY WYNIK:")
# print(f"Nasza sieć neuronowa myli się średnio o: {mae_w_dolarach:.2f} dolarów na każdym obrazie.")

In [166]:
# odnormalizowane_ceny_test = prawdziwa_cena(activation3.output)
# ujemne_ceny = odnormalizowane_ceny_test[odnormalizowane_ceny_test < 0]


In [167]:
# # Upewniamy się, że mamy obliczone przewidywania w prawdziwych dolarach
# wymyslone_ceny_dolary = prawdziwa_cena(activation3.output)
# prawdziwe_ceny_dolary = prawdziwa_cena(y_test)

# # 1. MAE (Mean Absolute Error) - Średni Błąd Bezwzględny
# mae = np.mean(np.abs(prawdziwe_ceny_dolary - wymyslone_ceny_dolary))

# # 2. RMSE (Root Mean Squared Error) - Pierwiastek Błędu Średniokwadratowego
# mse_dolary = np.mean((prawdziwe_ceny_dolary - wymyslone_ceny_dolary)**2)
# rmse = np.sqrt(mse_dolary)

# # 3. R^2 (Współczynnik determinacji)
# # Suma kwadratów reszt (błędów modelu)
# ss_res = np.sum((prawdziwe_ceny_dolary - wymyslone_ceny_dolary)**2)
# # Suma kwadratów całkowita (jakbyśmy zawsze zgadywali średnią cenę)
# ss_tot = np.sum((prawdziwe_ceny_dolary - np.mean(prawdziwe_ceny_dolary))**2)
# r2 = 1 - (ss_res / ss_tot)

# # 4. MAPE (Mean Absolute Percentage Error) - Średni Błąd Procentowy
# # Dodajemy małe zabezpieczenie przed dzieleniem przez zero (choć przy cenach raczej zera nie ma)
# non_zero = prawdziwe_ceny_dolary != 0
# mape = np.mean(np.abs((prawdziwe_ceny_dolary[non_zero] - wymyslone_ceny_dolary[non_zero]) / prawdziwe_ceny_dolary[non_zero])) * 100

# print("\n" + "="*40)
# print("RAPORT EWALUACJI MODELU (DANE TESTOWE):")
# print("="*40)
# print(f"1. MAE (Średnia pomyłka):     {mae:10.2f} $")
# print(f"2. RMSE (Kara za ekstrema):   {rmse:10.2f} $")
# print(f"3. Współczynnik R^2:          {r2:10.4f} (max 1.0)")
# print(f"4. MAPE (Błąd procentowy):    {mape:10.2f} %")
# print("="*40)